# BM25-OCR, BGE-OCR, TextRAG-top1, TextRAG-top3.

In [ ]:
!pip install -q datasets huggingface_hub rank_bm25 sentence-transformers
!pip install -q transformers accelerate qwen-vl-utils easyocr

## 1. Config

In [ ]:
import os, re, json, string, random, math, time, glob
import numpy as np
from google.colab import drive
drive.mount("/content/drive")

SET_NAME = "InfoVQA"            # "MP-DocVQA" or "InfoVQA"
REPO = {"MP-DocVQA": "openbmb/VisRAG-Ret-Test-MP-DocVQA",
        "InfoVQA":   "openbmb/VisRAG-Ret-Test-InfoVQA"}[SET_NAME]
CACHE = f"/content/drive/MyDrive/visrag/{SET_NAME}"
os.makedirs(CACHE, exist_ok=True)
N_QUESTIONS, SEED = 200, 0

# ONE prompt for ALL systems
PROMPT = ("Answer the question using the provided document. "
          "Reply with only the short answer, no explanation.\n"
          "Question: {q}")

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


## 2. Data (corpus, queries, qrels, fixed 200-question subset)

In [ ]:
import csv
from datasets import load_dataset
from huggingface_hub import list_repo_files, hf_hub_download

def load_beir_qrels(path):
    qrels = {}
    with open(path) as f:
        for row in csv.DictReader(f, delimiter="\t"):
            qrels.setdefault(row["query-id"], {})[row["corpus-id"]] = int(row["score"])
    return qrels

corpus  = load_dataset(REPO, name="corpus",  split="train")
queries = load_dataset(REPO, name="queries", split="train")
qf = [f for f in list_repo_files(REPO, repo_type="dataset")
      if f.startswith("qrels/") and f.endswith(".tsv")]
assert len(qf) == 1, qf
qrels_raw = load_beir_qrels(hf_hub_download(REPO, qf[0], repo_type="dataset"))

print(corpus[0].keys(), queries[0])
page_ids = [str(r["corpus-id"]) for r in corpus]
qrels = {qid: {pid for pid, s in d.items() if s > 0} for qid, d in qrels_raw.items()}
q_all = [{"qid": str(r["query-id"]), "question": r["query"],
          "answers": r["answer"] if isinstance(r["answer"], list) else [r["answer"]]}
         for r in queries]

sub_path = f"{CACHE}/subset_qids.json"
if os.path.exists(sub_path):
    keep = set(json.load(open(sub_path)))
else:
    random.seed(SEED)
    keep = {q["qid"] for q in random.sample(q_all, min(N_QUESTIONS, len(q_all)))}
    json.dump(sorted(keep), open(sub_path, "w"))
q_sub = [q for q in q_all if q["qid"] in keep]
print(len(page_ids), "pages |", len(q_all), "questions |", len(q_sub), "in subset")
print("qrels pages found in corpus:", len({p for s in qrels.values() for p in s} & set(page_ids)))

dict_keys(['corpus-id', 'image']) {'query-id': '36966.jpeg-1', 'query': 'Which social media site was invented in 2002?', 'answer': ['friendster'], 'options': None, 'is_numerical': 0}
459 pages | 718 questions | 200 in subset
qrels pages found in corpus: 459


## 3. OCR with EasyOCR (GPU)

In [ ]:
import torch, easyocr
print("GPU available:", torch.cuda.is_available())
reader = easyocr.Reader(["en"], gpu=True)

def ocr_page(pil_img):
    lines = reader.readtext(np.array(pil_img.convert("RGB")), detail=0, paragraph=False)
    return "\n".join(lines)

def run_ocr():
    path = f"{CACHE}/ocr.json"
    texts = json.load(open(path)) if os.path.exists(path) else {}
    todo = [i for i, pid in enumerate(page_ids) if not texts.get(pid, "").strip()]
    print(f"{len(todo)} pages to OCR")
    t0 = time.time()
    for n, i in enumerate(todo):
        txt = ocr_page(corpus[i]["image"])
        if txt.strip():
            texts[page_ids[i]] = txt
        else:
            print("WARNING: empty OCR for", page_ids[i])
        if n % 25 == 0:
            json.dump(texts, open(path, "w"))
            print(f"{n}/{len(todo)}  {(time.time()-t0)/60:.1f} min", end="\r")
    json.dump(texts, open(path, "w"))
    return texts

ocr_text = run_ocr()
docs = [ocr_text.get(p, "") for p in page_ids]
print("\nempty pages:", sum(not d.strip() for d in docs), "of", len(docs))

# free the GPU before loading the generator
del reader
import gc; gc.collect(); torch.cuda.empty_cache()

GPU available: True
357 pages to OCR

empty pages: 0 of 459


## 4. Retrievers and metrics

In [ ]:
def tok(s): return re.findall(r"\w+", s.lower())

def bm25_rankings():
    from rank_bm25 import BM25Okapi
    bm = BM25Okapi([tok(d) for d in docs])
    return {q["qid"]: [page_ids[i] for i in np.argsort(-bm.get_scores(tok(q["question"])))[:10]]
            for q in q_all}

def bge_rankings():
    from sentence_transformers import SentenceTransformer
    m = SentenceTransformer("BAAI/bge-small-en-v1.5", device="cuda")
    D = m.encode(docs, batch_size=64, normalize_embeddings=True, show_progress_bar=True)
    inst = "Represent this sentence for searching relevant passages: "
    Q = m.encode([inst + q["question"] for q in q_all], normalize_embeddings=True)
    S = Q @ D.T
    out = {q["qid"]: [page_ids[i] for i in np.argsort(-S[j])[:10]] for j, q in enumerate(q_all)}
    del m; gc.collect(); torch.cuda.empty_cache()
    return out

def retrieval_metrics(rank, qrels, qids):
    r1 = r5 = mrr = ndcg = 0.0
    for qid in qids:
        rel, top = qrels[qid], rank[qid]
        r1 += any(d in rel for d in top[:1])
        r5 += any(d in rel for d in top[:5])
        for k, d in enumerate(top[:10]):
            if d in rel:
                mrr += 1 / (k + 1); break
        dcg  = sum(1 / math.log2(k + 2) for k, d in enumerate(top[:5]) if d in rel)
        idcg = sum(1 / math.log2(k + 2) for k in range(min(len(rel), 5)))
        ndcg += dcg / idcg
    n = len(qids)
    return {"Recall@1": r1/n, "Recall@5": r5/n, "MRR@10": mrr/n, "nDCG@5": ndcg/n}

def norm(s):
    s = s.lower()
    s = "".join(c for c in s if c not in string.punctuation)
    s = re.sub(r"\b(a|an|the)\b", " ", s)
    return " ".join(s.split())

def correct(pred, golds):
    return norm(pred) in {norm(g) for g in golds}

## 5. Generator (shared)

In [ ]:
def load_generator():
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor
    name = "Qwen/Qwen2.5-VL-3B-Instruct"
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name, torch_dtype=torch.float16, device_map="auto")      # T4: fp16, not bf16
    proc = AutoProcessor.from_pretrained(
        name, min_pixels=256*28*28, max_pixels=1024*28*28)       # lower max_pixels if OOM
    return model, proc

def generate(model, proc, question, texts=None, images=None, max_new_tokens=32):
    content = []
    for t in (texts or []):
        content.append({"type": "text", "text": f"Document:\n{t}"})
    for im in (images or []):
        content.append({"type": "image", "image": im})
    content.append({"type": "text", "text": PROMPT.format(q=question)})
    prompt = proc.apply_chat_template([{"role": "user", "content": content}],
                                      tokenize=False, add_generation_prompt=True)
    inputs = proc(text=[prompt], images=images or None, return_tensors="pt",
                  padding=True).to(model.device)
    out = model.generate(**inputs, max_new_tokens=max_new_tokens, do_sample=False)
    return proc.batch_decode(out[:, inputs.input_ids.shape[1]:],
                             skip_special_tokens=True)[0].strip()

def run_system(model, proc, tag, build_inputs):
    """Generic runner: build_inputs(q) -> dict(texts=..., images=...). Cached per question."""
    path = f"{CACHE}/gen_{tag}.json"
    preds = json.load(open(path)) if os.path.exists(path) else {}
    for q in q_sub:
        if q["qid"] in preds: continue
        preds[q["qid"]] = generate(model, proc, q["question"], **build_inputs(q))
        if len(preds) % 20 == 0: json.dump(preds, open(path, "w"))
    json.dump(preds, open(path, "w"))
    acc = float(np.mean([correct(preds[q["qid"]], q["answers"]) for q in q_sub]))
    print(f"{tag}: accuracy = {acc:.3f}")
    return preds, acc

## 6. Task 1

In [ ]:
sub_ids = [q["qid"] for q in q_sub]
rankings = {"BM25-OCR": bm25_rankings(), "BGE-OCR": bge_rankings()}

results = {}
for n, r in rankings.items():
    results[n] = {"subset200": retrieval_metrics(r, qrels, sub_ids),
                  "all":       retrieval_metrics(r, qrels, [q["qid"] for q in q_all])}
    print(n, {k: round(v, 3) for k, v in results[n]["subset200"].items()})
json.dump(results, open(f"{CACHE}/retrieval_text.json", "w"), indent=2)

best = max(results, key=lambda n: results[n]["subset200"]["nDCG@5"])
print("Selected text retriever:", best)
json.dump(rankings[best], open(f"{CACHE}/best_text_rank.json", "w"))

# sanity: Recall@5 should be well above chance (~0.7%)
assert results[best]["subset200"]["Recall@5"] > 0.3, "retrieval looks broken, check OCR"

Loading weights:   0%|          | 0/199 [00:00<?, ?it/s]

Batches:   0%|          | 0/8 [00:00<?, ?it/s]

BM25-OCR {'Recall@1': 0.66, 'Recall@5': 0.81, 'MRR@10': 0.715, 'nDCG@5': 0.735}
BGE-OCR {'Recall@1': 0.69, 'Recall@5': 0.89, 'MRR@10': 0.772, 'nDCG@5': 0.796}
Selected text retriever: BGE-OCR


In [ ]:
import os
import gc
import torch

# Install bitsandbytes to enable 4-bit loading
!pip install -q bitsandbytes

# Configure PyTorch memory allocator
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

# Force cleanup of any previous models
if 'model' in globals():
    del model
if 'proc' in globals():
    del proc
gc.collect()
torch.cuda.empty_cache()

def load_generator_4bit():
    from transformers import Qwen2_5_VLForConditionalGeneration, AutoProcessor, BitsAndBytesConfig
    name = "Qwen/Qwen2.5-VL-3B-Instruct"
    nf4_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_quant_type="nf4",
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.float16
    )
    model = Qwen2_5_VLForConditionalGeneration.from_pretrained(
        name,
        quantization_config=nf4_config,
        device_map="auto"
    )
    proc = AutoProcessor.from_pretrained(name, min_pixels=256*28*28, max_pixels=1024*28*28)
    return model, proc

model, proc = load_generator_4bit()
rank = rankings[best]

# Load existing results to avoid losing top1 progress
acc_path = f"{CACHE}/acc_task1.json"
if os.path.exists(acc_path):
    try:
        accs = json.load(open(acc_path))
    except Exception:
        accs = {}
else:
    accs = {}

def get_truncated_texts(q, k=3):
    truncated_docs = []
    for p in rank[q["qid"]][:k]:
        doc_text = ocr_text.get(p, "")
        words = doc_text.split()
        if len(words) > 800:
            doc_text = " ".join(words[:800])
        truncated_docs.append(doc_text)
    return {"texts": truncated_docs}

# Run only TextRAG-top3
with torch.no_grad():
    gc.collect()
    torch.cuda.empty_cache()
    tag = "TextRAG-top3"
    _, accs[tag] = run_system(model, proc, tag, lambda q: get_truncated_texts(q, k=3))

json.dump(accs, open(acc_path, "w"), indent=2)

Loading weights:   0%|          | 0/824 [00:00<?, ?it/s]

TextRAG-top3: accuracy = 0.290
